# 4 — Full Pipeline Integration

Tests the integrated system with both components (Bingham filter + T³ QAN) on the 2D baseline arena (not running the experiment).

In [1]:
!pip install torch numpy matplotlib
!pip install git+https://github.com/jacobsvennevik/MADE.git


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
  Cloning https://github.com/jacobsvennevik/MADE.git to /private/var/folders/rr/v38_ky511jl__2hdj2j5ggc40000gn/T/pip-req-build-xhl70d1p
  Running command git clone --filter=blob:none --quiet https://github.com/jacobsvennevik/MADE.git /private/var/folders/rr/v38_ky511jl__2hdj2j5ggc40000gn/T/pip-req-build-xhl70d1p
  Resolved https://github.com/jacobsvennevik/MADE.git to commit 1673eeabd571b4815122c3948694fae13de9f039
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for ManifoldAttractors: filename=manifoldattractors-0.0.1-py3-none-any.whl size=24323 sha256=f40dc52f265322c35e34cbaa05dd47f2cddcff25414c67bb2f5b3d64e88c2499
  Stored in directory: /private/var/folders/rr/v38_ky511jl__2hdj2j5ggc40000gn/T/pip-ephem-wheel-cache-45c3i_sg/wheels/c1/5d/5d/73acadd7918657ad77d2082ed2c05790

In [2]:
import os
import sys
import time
import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, '/workspace/CAN_Path_Integration_3D_model')

from network.visualize3D import visualize_trajectory_projections, plot_pi_error
from experiments.arena_2d import Arena2DExperiment, Arena2DConfig
from analysis.scoring import score_2d
from metrics import wrapped_angle_diff
from config import RunConfig, NetworkConfig, ExperimentConfig
from experiments.arena_2d import Arena2DExperiment




ModuleNotFoundError: No module named 'network'

In [ ]:
# config + experiment
g_vec = np.array([0., 0., -9.81])
cfg = RunConfig(
    network=NetworkConfig(build_connectivity=False),                 # validated defaults
    experiment=Arena2DConfig(n_steps=100000), # Arena2DConfig → has run_name
)
exp = Arena2DExperiment(cfg, record=True)    # config, not g


## Set up the arena

In [ ]:
scale = cfg.experiment.scale
print("scale:", scale, " speed:", cfg.experiment.speed,
      " target_speed_rad:", getattr(cfg, "target_speed_rad", None))
print("commanded theta_dot = scale*speed =", scale * cfg.experiment.speed, "rad/step")
print("ceiling = 0.0168  -> saturated?" , scale*cfg.experiment.speed > 0.0168)

In [ ]:
# run + save
print("Running ...")
t0 = time.time()
result = exp.run_experiment(g=g_vec)
exp.save(result, f"workspace/runs/{cfg.experiment.run_name}.npz")
print(f"Saved runs/{cfg.experiment.run_name}.npz  ({time.time()-t0:.1f}s)")

# everything from the result object (single source of truth)
world_pos  = result.world_pos
torus_gt   = result.torus_gt
theta_hist = result.theta_hist
gap        = result.gap_hist
n_hat_hist = result.n_hat_hist
T          = cfg.experiment.n_steps

print(f"Filter gap  t=100: {gap[99]:.4f}  t=500: {gap[499]:.4f}  t={T}: {gap[-1]:.4f}  target > 2 by t=500")
print(f"n_hat at t={T}: {n_hat_hist[-1]}  (target [0, 0, 1])")
print(f"theta_3 decoded range: [{theta_hist[:, 2].min():.4f}, {theta_hist[:, 2].max():.4f}]  target ~[0, 0]")
print(f"MADE error  full: {result.mean_norm_error:.4f}  after t=200: {result.norm_error[200:].mean():.4f}")

In [ ]:
import numpy as np
from metrics import wrapped_angle_diff

m  = exp.qan.manifold.metric
sc = cfg.experiment.scale

W = 200
gt_d  = wrapped_angle_diff(torus_gt[W:],   torus_gt[:-W])[:, :2]
dec_d = wrapped_angle_diff(theta_hist[W:], theta_hist[:-W])[:, :2]
sg, sd = np.linalg.norm(gt_d,1), np.linalg.norm(dec_d,1)
print("windowed cos:", np.median((dec_d*gt_d).sum(1)/(sd*sg+1e-9)))

# (a) the two scale candidates + what to_phase actually is (linear? a twist matrix?)
print("scale used       :", sc)
print("2pi/grid_spacing :", 2*np.pi/cfg.experiment.grid_spacing, " <- correct")
print("2pi/env_size     :", 2*np.pi/cfg.experiment.env_size,     " <- 'bug' value")
for vv in (np.array([[1.,0.]]), np.array([[0.,1.]]), np.array([[2.,0.]])):
    print("  to_phase", vv.ravel(), "->", np.round(m.to_phase(vv).ravel(), 4))

# (b) did the bump integrate at the right SPEED and DIRECTION? -> tests BOTH pasted bugs
gt_vel  = wrapped_angle_diff(torus_gt[1:],   torus_gt[:-1])[:, :2]   # how it SHOULD move
dec_vel = wrapped_angle_diff(theta_hist[1:], theta_hist[:-1])[:, :2] # how it ACTUALLY moved
sp_gt, sp_dec = np.linalg.norm(gt_vel,1), np.linalg.norm(dec_vel,1)
mask = sp_gt > 1e-4
print("speed ratio decoded/gt :", round(np.median(sp_dec[mask]/sp_gt[mask]),3),
      " (1=correct, ~.15=6.7x bug, ~.88=gain65)")
cos = (dec_vel*gt_vel).sum(1)/(sp_dec*sp_gt+1e-12)
print("direction cos(dec,gt)  :", round(np.median(cos[mask]),3),
      " (1=aligned, <1=mis-rotated => twist)")

# (c) what KIND of tracking error: constant offset vs accumulating drift?
e = np.linalg.norm(wrapped_angle_diff(theta_hist, torus_gt), axis=1)
print("err t<500:", round(e[:500].mean(),3), "| t~20k:", round(e[19750:20250].mean(),3),
      "| t>39.5k:", round(e[-500:].mean(),3), "| mean as %period:", round(e.mean()/(2*np.pi),3))



In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
ax.plot(world_pos[:, 0], world_pos[:, 1], lw=0.5, alpha=0.7, color="steelblue")
ax.scatter(*world_pos[0, :2],  color="green", s=60, zorder=5, label="start")
ax.scatter(*world_pos[-1, :2], color="red",   s=60, zorder=5, label="end")
ax.set_xlabel("x (m)")
ax.set_ylabel("y (m)")
ax.set_title("Physical world trajectory (flat 2D arena)")
ax.set_aspect("equal")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
fig, _ = visualize_trajectory_projections(
    torus_gt, theta_hist,
    title="2D baseline arena, ground truth vs decoded on the torus manifold",
)
plt.show()


In [ ]:
fig, _ = plot_pi_error(torus_gt, theta_hist)
plt.show()

In [ ]:
out = score_2d(f"workspace/runs/{cfg.experiment.run_name}.npz", n_neuron=5000, norm="minmax")
hgs, sgs = out["hgs"], out["sgs"]
grid_like = np.isfinite(hgs) & (hgs > 0.3) & (sgs < hgs)
n_act  = out["f"].shape[-1]
active = np.isfinite(hgs)

print(f"occupancy      : {out['occupancy']:.0%}")
print(f"ring detected  : {out['ring_frac']:.0%} of active cells")
print(f"grid-like      : {grid_like.sum()}/{n_act}  (HGS>0.3 & SGS<HGS)")
if active.any():
    print(f"median HGS (all)      : {np.nanmedian(hgs):.2f}")
    if grid_like.any():
        print(f"median HGS (grid-like): {np.nanmedian(hgs[grid_like]):.2f}")
    print(f"grid-like / ACTIVE    : {grid_like.sum()}/{active.sum()} = {grid_like.sum()/active.sum():.0%}")

In [ ]:
order = np.argsort(np.where(np.isfinite(hgs), hgs, -np.inf))[::-1][:3]
fig, axes = plt.subplots(len(order), 2, figsize=(6, 2.6*len(order)))
axes = np.atleast_2d(axes)
for r, k in enumerate(order):
    axes[r,0].imshow(out["f"][..., k].T, origin="lower", cmap="viridis")
    axes[r,0].set_ylabel(f"cell {k}\nHGS={hgs[k]:.2f}", fontsize=9)
    axes[r,1].imshow(out["ac"][..., k].T, origin="lower", cmap="viridis")
    for c in (0,1): axes[r,c].set_xticks([]); axes[r,c].set_yticks([])
plt.tight_layout(); plt.show()

In [ ]:
BINS = 40
half = cfg.experiment.env_size / 2.0

xy   = world_pos[:, :2] / half
ij   = np.clip(np.floor((xy + 1.0) * 0.5 * BINS).astype(np.int64), 0, BINS - 1)
flat_indices = (ij[:, 0] * BINS + ij[:, 1]).tolist() 

bk  = exp.last_integrator.backend
acc = bk.allocate_ratemap(BINS)
for t in range(len(torus_gt)):
    bk.reset(torus_gt[t], radius=0.25)     
    bk.record_ratemap(acc, flat_indices[t])
sums_or, counts_or = bk.ratemap_to_numpy(acc, BINS)

In [ ]:
integ = exp.last_integrator
integ.reset(torus_gt[0])
integ.warmup(100)

theta_hist_rm = integ.run(
    result.v_body_seq, g_vec,
    flat_indices=np.array(flat_indices),
    ratemap_bins=BINS,
)
h = {k: list(v) for k, v in integ.history.items()}   # snapshot before oracle clears it

sums_real, counts_real = integ.ratemap_sums, integ.ratemap_counts

In [ ]:
from analysis.scoring import score_2d_from_map

def _report(tag, sums, counts):
    r  = score_2d_from_map(sums, counts)
    gl = r["grid_like"]
    print(f"{tag}: occ {r['occupancy']:.0%} | ring {r['ring_frac']:.0%} | "
          f"grid-like {gl.sum()}/{len(r['hgs'])} ({100*gl.mean():.1f}%) | "
          f"median HGS {np.nanmedian(r['hgs']):.3f} | active {r['n_active']}")
    print(f"       SGS≥HGS (square confusion): "
          f"{(np.isfinite(r['sgs']) & (r['sgs'] >= r['hgs'])).sum()} cells")
    return r

out_real   = _report("real  ", sums_real,   counts_real)
out_oracle = _report("oracle", sums_or, counts_or)

if h["z2"]:
    gap = np.array(h["z2"]) - np.array(h["z1"])
    print("filter gap end:", gap[-1], "  n_hat end:", np.array(h["n_hat"])[-1])